- 38:06

# Roadmap

- linux
- sql
- java /python/scala
- bigdata  ( hdfs,hive->(hadoop), batch,spark sql->(spark) )
- ETL concepts
- cloud
- work on projects, do optimizations

### File system

- windows (NTFS - new technology file system )
- linux (EXT-extensible file system)
- mac (macfs)

- if read/write data , it goes through file system from/to  hard disk

- NTFS,EXT --> Standalone file system
- HDFS,S3  --> Distributed file system

- 
   - **distributed** computing on the basis of data engineer and network/infrastructure are different
   - in n/w concept ,they consider toplology or physical structure , where different machines connected by a lan or other network can be said to be distributed
   - for data engineering perspective , data need to be distributed to call it as a distributed system

- Block 
  - if we want to store a 1 gb data , it will be split it blocks and store
  - NTFS - 16 KB
  - EXT  - 512 kb
  
  

### Types of distributed system

![](distri.jpg)

- master slave (hadoop,spark)
- peer to peer (cassandra)
-
- in master slave architecture there wont be any communication between slaves
- in peer to peer model eanch and every node know the status of other node.


- SPOF (Single point of failure)
  - reason is SPOC(Single point of communication)

___

-
- H0
- H1
-
- H2
- H3
-
-
- H0,H1 have more similar architecture, and H2,H3 is also similar
- between these two there is difference



___

- a programm in excecution is process
- a background process - daemon process
- 
-  hadoop with 5 daemon process (jp1,jp2,jp3,jp4,jp5)(JAVA PROCESS)
   - first 3 for hdfs ,next 2 for mapreduce
    

- hadoop hdfs have default replication technology(3 replicas)
- 

___

1. **NameNode (HDFS Master)**
   - Manages the file system namespace.
   - Keeps metadata (like filenames, block locations).
   - Does not store actual data — only tracks where data lives.
   - There’s typically only one active NameNode per cluster.
2. **DataNode (HDFS Slave)**
   - Stores the actual data blocks.
   - Periodically sends heartbeats and block reports to the NameNode.
   - Can be many DataNodes in a cluster.
3. **ResourceManager (YARN Master)**
   - Manages resources across the cluster.
   - Schedules and allocates containers (memory, CPU) to applications.
   - Works with NodeManagers.
4. **NodeManager (YARN Slave)**
   - Runs on each DataNode.
   - Reports node health and resource usage to the ResourceManager.
   - Manages containers on that node (where tasks run).
5. **Secondary NameNode (Checkpointing Assistant)**
   - Does NOT replace the NameNode during failure.
   - Periodically merges the file system image (FsImage) and edit logs.
   - Reduces load on NameNode by creating checkpoint copies of metadata.




| Daemon               | Role                 | Component | Runs On         |
|----------------------|----------------------|-----------|-----------------|
| **NameNode**         | Stores metadata      | HDFS      | Master Node     |
| **DataNode**         | Stores actual data   | HDFS      | Slave Nodes     |
| **ResourceManager**  | Manages resources    | YARN      | Master Node     |
| **NodeManager**      | Manages containers   | YARN      | Slave Nodes     |
| **Secondary NameNode** | Checkpointing     | HDFS      | Separate Node (optional) |


___


 ![](master_to_process_jp1.png)

- all slaves run(JP2)
- the node which run JP1 hadoop process is master
- the node which run JP2 hadoop process is slave

- how to connect to cluster to read and write
- in real time they will not allow you to directly connect to node 
   - they will create a firewall kind of node **(Edge node)**

- if you are creating multinode cluster on your own then can connect slave machine without edge node concept



![](edge.png)

___

- consider you are using facebook
  - the device in which you operate is edge device
  - the data we upload from edge node will be distributed on slave nodes,and wont come back to edge node
 

### read write architecture

 - after installing hadoop we have two file system
    - EXT,HDFS (distributed system on top of standalone file system)
 - use specific hadoop commands to upload /down load   files with hdfs

- write 
   - send data to a Ext -> load to hdfs by cmd
   - 
   - a process called **hadoop client api** will get started on top of node
   - client api will redirect this request to master node
      -  data is still in local node , masternode have file name,size
      - master create meta data , for write rquest 
      - meta data: block size,no of replicas, placement allocation (where do these replicas will be kept)(rack awareness policy)
      - meta data will be **stored in ext** of master machine
      - master machine replays to client machine with meta data (to place data blocks in respective nodes)
      - client api will create a seperate process called pipeline -> will split data to blocks and distribute
      - once everything is complete there will be a heartbeat communication back to each nodes (acknowledgement)
       

- whether there is no read and write operation happening or not ,there will be communication between slave and master within every 3 seconds (heart beat)


- **Data**
   - Block (Slave)
   - Metadata (master)
- **Node**
   - Slave
   - Master

- Let there is a **Failure** in write statement
  - network/software failure(temporary)
  - hardware failure(permanent)
  

- automatic failure
   - hadoop tries to satisfy no of copies all time
   - try to copy the data to another node
   

- for temporary failure
   - master will recreate these blocks in another node ,delete the temporary lost node and create it as a new node. 

- when master node goes down ,all jobs(read/write/hive/mr) will killed immediatly
- client asks whether HA(high availability) 

- For **Read** operation , there is no need for pipeline , client API send request to master node and master responds with meta data , client api directly meets slave node
- when perform read request to master node ,it replies with details of multiple copeis ,so if one is missed for read , wont complain but read next


- some error happens while writing
- the write request of pipeline wont be affected
   - so the pipeline will send the -ve acknowledgement to master ,so master will update meta data, and pipeline will continue.

- read/write fails if
  - all nodes fails
  - client api killed
  

___

- in hadoop 2 they implemented HA
   - high availabilty(more than one master)
   - JP1 ->Namenode
   - JP2 -> DataNode
   - JP3 -> Secondary Namenode 
   <br>
    **MAPREDUCE**   
   - JP4 -> jobtracker (V1) <BR>
         ->Resource manager(V2)
   - JP5 ->taskTracker(v1)<BR>
         ->NodeManager(v2)

- One **active** namenode and N **passive** namenodes
- cluster coordinator technology **Zookeeper** is behind choosing a new namenode when active namenode is down
- zookeeper (multi node) is another cluster (not client server/peer to peer but **Leader follower**)
- even passive namenode recieves heart beat

- they keep meta data in a centralized machine and called **journal node**.
- High availability : 2 NameNodes(active+**standby**) work together , and journalNode help them sync their metadata
- When the active NameNode makes changes to the file system (like creating files or directories), it writes those changes to the **EditLog**.
- These EditLogs are written to a set of JournalNodes.
- The standby NameNode reads these edits from the JournalNodes to stay in sync.


___

### Types of node

#### H1
 - master node (name node+job tracker)
 - slave node (data node+task tracker)
 - checkpoint node(secondary node)

### H2 without HA
- master node (name node +resource manager)
- slave node (datanode + node manager)
- secondary name node

### H1
1. single node (pseudo node)- for testing or learning only
2. Distributed cluster
    - atleast 5 node (Master(nameN+Jobtracker),SecondaryNN,3 SLAVE for keeping 3 replicas)
    - it is possible with 3 nodes (One machine can implement slave node along with master node)
    - or can have a slave on SNN
    - name node and job tracker can be in teo nodes ,combination of both is Master
    - but cannot split data node and task tracker
    

### H2

 - Job tracker is renamed as resource manager
-  **with HA passive name node will take care of secondary name node**,no separate secondary name node but only passive name node
- if install h2 without HA, secondary name node will be created automatically 
 

- Master Daemon - (NameNode+JobTracker)
- Slave Daemon - (DataNode+Tasktracker)
- CheckPoint Node - Secondary name node